In [ ]:
import xarray as xr
import numpy as np
from pathlib import Path
import pandas as pd
import warnings
import proplot as pplt

In [ ]:
def vector_norm(x, dim, ord=None):
    return xr.apply_ufunc(
        np.linalg.norm, x, input_core_dims=[[dim]], kwargs={"ord": ord, "axis": -1}
    )

In [ ]:
output_dir_def_par = Path("output/top_ensemble")
output_dir_reg_par = Path("output/top_ensemble_reg")

In [ ]:
def preprocess_oifs_ensemble(ds: xr.Dataset) -> xr.Dataset:
    source_file = Path(ds.encoding["source"])
    if "reg" in str(source_file):
        albedo_scheme = "regularized"
    else:
        albedo_scheme = "default"

    start_date = pd.Timestamp(source_file.parent.parent.name.replace("_", ", "))
    ds = ds.expand_dims(
        albedo_scheme=[albedo_scheme],
        start_date=[start_date],
    )
    try:
        ds = ds.drop_vars("ncextr")
    except ValueError:
        pass
    return ds

def preprocess_nemo_ensemble(ds: xr.Dataset) -> xr.Dataset:
    source_file = Path(ds.encoding["source"])
    if "reg" in str(source_file):
        albedo_scheme = "regularized"
    else:
        albedo_scheme = "default"

    start_date = pd.Timestamp(source_file.parent.parent.name.replace("_", ", "))
    ds = ds.isel(y=0, x=0)
    ds = ds.rename(time_counter="time")
    ds = ds.convert_calendar("gregorian")
    ds = ds.assign_coords(time=ds.time.data - np.datetime64(start_date.date()))
    ds = ds.expand_dims(
        albedo_scheme=[albedo_scheme],
        start_date=[start_date],
    )
    return ds


In [ ]:
experiment_directories = []
for date_dir in output_dir_def_par.glob("*"):
    experiment_directories.append(date_dir / "iter_1")
for date_dir in output_dir_reg_par.glob("*"):
    experiment_directories.append(date_dir / "parallel")

progvars_ensemble = [
    experiment_dir / "progvar.nc" for experiment_dir in experiment_directories
]
diagvars_ensemble = [
    experiment_dir / "diagvar.nc" for experiment_dir in experiment_directories
]
nemo_t_ensemble = [
    next(experiment_dir.glob("*_grid_T*.nc"))
    for experiment_dir in experiment_directories
]
ice_ensemble = [
    next(experiment_dir.glob("*_icemod*.nc"))
    for experiment_dir in experiment_directories
]

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    diag_ensemble = xr.open_mfdataset(
        diagvars_ensemble, preprocess=preprocess_oifs_ensemble
    )
diag_forecast = diag_ensemble.isel(time=-1)
diag_forecast = diag_forecast.assign_coords(
    start_date=diag_forecast.start_date + diag_forecast.time
)
diag_forecast = diag_forecast.rename(start_date="end_date")

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prog_ensemble = xr.open_mfdataset(
        progvars_ensemble, preprocess=preprocess_oifs_ensemble
    )
prog_forecast = prog_ensemble.isel(time=-1)
prog_forecast = prog_forecast.assign_coords(
    start_date=prog_forecast.start_date + prog_forecast.time
)
prog_forecast = prog_forecast.rename(start_date="end_date")

In [ ]:
print(float(diag_forecast.pbl_height.mean()))
print(float(diag_forecast.pbl_height.std()))
print(float(prog_forecast.height_f.sel(nlev=120).mean()))
print(float(prog_forecast.height_f.sel(nlev=120).std()))

In [ ]:
prog_forecast_diff = prog_forecast.sel(albedo_scheme="regularized") - prog_forecast.sel(albedo_scheme="default")

In [ ]:
t_pbl_diff = prog_forecast_diff.t.sel(nlev=range(120, 138))
t_pbl_diff_l2 = vector_norm(t_pbl_diff.load(), "nlev", 2)
print(float(t_pbl_diff_l2.max()))
print(float(t_pbl_diff_l2.mean()))

In [ ]:
q_pbl_diff = prog_forecast_diff.q.sel(nlev=range(120, 138)) * 1e3
q_pbl_diff_l2 = vector_norm(q_pbl_diff.load(), "nlev", 2)
print(float(q_pbl_diff_l2.max()))
print(float(q_pbl_diff_l2.mean()))

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    nemo_ensemble = xr.open_mfdataset(
        nemo_t_ensemble, preprocess=preprocess_nemo_ensemble
    )

In [ ]:
nemo_forecast = nemo_ensemble.isel(time=-1)
nemo_forecast = nemo_forecast.assign_coords(
    start_date=nemo_forecast.start_date + nemo_forecast.time
)
nemo_forecast = nemo_forecast.rename(start_date="end_date")
sst_diff = np.abs(
    nemo_forecast.sel(albedo_scheme="regularized") - nemo_forecast.sel(albedo_scheme="default")
).sosstsst
print(float(sst_diff.max()))
print(float(sst_diff.mean()))

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ice_ensemble = xr.open_mfdataset(
        ice_ensemble, preprocess=preprocess_nemo_ensemble
    )

In [ ]:
ice_forecast = ice_ensemble.isel(time=-1)
ice_forecast = ice_forecast.assign_coords(
    start_date=ice_forecast.start_date + ice_forecast.time
)
ice_forecast = ice_forecast.rename(start_date="end_date")
ist_diff = np.abs(
    ice_forecast.sel(albedo_scheme="regularized") - ice_forecast.sel(albedo_scheme="default")
).icettop
print(float(ist_diff.max()))
print(float(ist_diff.mean()))

In [ ]:
fig, axs = pplt.subplots(
    width="50em", height="55em", sharey=0, sharex=3, nrows=4
)

ims = []

ax = axs[0]
im = (prog_forecast.t - 273.15).sel(albedo_scheme="default", nlev=137).plot(ax=ax, label="default", color="k")
ims.append(im)
im = (prog_forecast.t - 273.15).sel(albedo_scheme="regularized", nlev=137).plot(ax=ax, label="regularized", color="k", ls="--")
ims.append(im)
ax.format(title="10m Temperature", ylabel="Temperature [°C]")

ax = axs[1]
nemo_forecast.sosstsst.sel(albedo_scheme="default").plot(ax=ax, label="default", color="k")
nemo_forecast.sosstsst.sel(albedo_scheme="regularized").plot(ax=ax, label="regularized", color="k", ls="--")
ax.format(title="Sea Surface Temperature", ylabel="Temperature [°C]")

ax = axs[2]
ice_forecast.icettop.sel(albedo_scheme="default").plot(ax=ax, label="default", color="k")
ice_forecast.icettop.sel(albedo_scheme="regularized").plot(ax=ax, label="regularized", color="k", ls="--")
ax.format(title="Sea Ice Surface Temperature", ylabel="Temperature [°C]")

ax = axs[3]
albedo = ice_forecast.iceconc_cat.sel(ncatice=1) * ice_forecast.icealb_cat.sel(ncatice=1)
for category in range(2, 6):
    albedo += ice_forecast.iceconc_cat.sel(ncatice=category) * ice_forecast.icealb_cat.sel(
        ncatice=category
    )
albedo.sel(albedo_scheme="default").plot(ax=ax, label="default", color="k")
albedo.sel(albedo_scheme="regularized").plot(ax=ax, label="regularized", color="k", ls="--")
ax.format(title="Sea Ice Albedo", ylabel="Albedo [-]")


axs.format(xlabel="Time", abc="a)")
fig.format(suptitle="Final Time Step for Varying Albedo Parameterization (Parallel Algorithm)")
fig.legend(ims, loc="b", title="Albedo Parameterization")
fig.savefig("top_albedo_difference_final_step_parallel.pdf")